# Epicollect5 -> tables, photos, GeoPackage

Pulls everything out of the Epicollect5 project named in `config.ini` and turns it
into products that can be pushed to ArcGIS Online every day.

| Step | Output |
|---|---|
| 1. Read `config.ini`, authenticate | OAuth token (cached on disk) |
| 2. Read the project structure | every form, branch, question and its type |
| 3. Download every entry | one CSV per form / branch + `pandas` DataFrames |
| 4. Download every photo | `photos/` folder, resumable |
| 5. Publish the photos | `git add / commit / push` into the photo repository |
| 6. Build public photo URLs | one URL column per photo question, for the ArcGIS popup |
| 7. Write a GeoPackage | one point layer per form, EPSG:4326, ArcGIS-safe field names |

**Re-running is cheap and safe.** Photos already on disk are skipped, the OAuth
token is reused until it expires, and the GeoPackage is rebuilt from scratch so
edits and deletions made in Epicollect5 propagate.

### API facts this notebook relies on

All of the following was measured against this project, not taken from the docs.

* Token endpoint `POST /api/oauth/token` with `grant_type=client_credentials`.
  **10 tokens per hour per IP**, each valid for 2 hours, so the token is cached.
* Entries and project-definition endpoints allow about **5 requests per minute**.
* Media allows exactly **10 requests per minute**, keyed by project slug rather
  than by IP, and a 429 carries **no `Retry-After` header**. This is what makes the
  first photo download take hours; later runs only fetch what is new.
* `map_index` does **not** default to `EC5_AUTO`. It defaults to whatever mapping
  the project has marked default, which can rename questions and even hide them.
  `MAP_INDEX = 0` is passed on every request so the column names never move.
* A **location** answer is a nested object
  `{"latitude", "longitude", "accuracy", "UTM_Northing", "UTM_Easting", "UTM_Zone"}`.
* A **photo** answer is a bare filename on a private project, and a full media URL
  on a public one. `media_filename` accepts either.
* Photos must be fetched from `/api/export/media/`. The `/api/media/` path that
  appears inside public entry JSON **returns 404 on a private project** even with a
  valid bearer token, because it authenticates by session rather than by OAuth.
* A **branch** question exports on the parent row only as a *count*; the branch
  answers themselves are a separate request (`branch_ref=...`) and come back keyed
  by `ec5_branch_owner_uuid`.
* A **missing media file still returns HTTP 200** with a 512x512 "not synced yet"
  placeholder JPEG. It is detected below, otherwise the photo folder quietly fills
  up with placeholders.
* An **expired token does not return 401**; it returns 404 with `ec5_256`, the same
  status as a project that does not exist. `api_get` matches on the error code.

## 1. Settings

Everything you need to change lives in this cell.

In [ ]:
from pathlib import Path

# ---------------------------------------------------------------- input ----
CONFIG_PATH = Path(
    r"D:\OneDrive_Emory\OneDrive - Emory\Research_doc\larvae\code\download_Epicollect5\config.ini"
)

# --------------------------------------------------------------- output ----
OUT_DIR = Path(
    r"D:\OneDrive_Emory\OneDrive - Emory\Research_doc\larvae\code\download_Epicollect5\output"
)
TABLE_DIR = OUT_DIR / "tables"          # one CSV per form / branch
GPKG_PATH = OUT_DIR / "epicollect5.gpkg"  # the file uploaded to ArcGIS Online

# ------------------------------------------------- photos + GitHub repo ----
# Point GITHUB_REPO_DIR at your local clone of the repository that serves the
# photos. Photos are written straight into it so `git push` is the only extra
# step. Leave it as None to keep photos under OUT_DIR instead.
GITHUB_REPO_DIR = None  # e.g. Path(r"D:\OneDrive_Emory\...\larvae-photos")
GITHUB_OWNER = "gladcolor"    # <-- your GitHub user or organisation
GITHUB_REPO = "larvae-photos"  # <-- the repository name
GITHUB_BRANCH = "main"
GITHUB_SUBDIR = "photos"       # folder inside the repository

# "raw"   -> https://raw.githubusercontent.com/<owner>/<repo>/<branch>/<subdir>/<file>
# "pages" -> https://<owner>.github.io/<repo>/<subdir>/<file>   (needs GitHub Pages on)
PHOTO_URL_STYLE = "raw"

PHOTO_DIR = (
    Path(GITHUB_REPO_DIR) / GITHUB_SUBDIR if GITHUB_REPO_DIR else OUT_DIR / "photos"
)

# ------------------------------------------------------------ behaviour ----
DOWNLOAD_PHOTOS = True
PHOTO_FORMAT = "entry_original"   # "entry_original" (1024x768) or "entry_thumb" (100x100)
MAX_PHOTOS = None                 # cap for a quick test run, e.g. 20; None = all

MAP_INDEX = 0        # 0 = EC5_AUTO. Gives stable "1_Question" column names and
                     # never hides a question, unlike a custom mapping.

# Leave False. ArcGIS Online can only *overwrite* a hosted feature layer while the
# schema is unchanged, and a question that happens to be blank on a given day would
# otherwise make its column disappear and break the next upload.
DROP_EMPTY_COLUMNS = False
PER_PAGE = 500       # API maximum
ENTRIES_PER_MINUTE = 4    # published limit is 5/min; stay just under it

# Measured against this project: /api/export/media allows exactly 10 requests per
# minute, after which it returns 429 ec5_255 with no Retry-After header. The bucket
# is keyed by PROJECT SLUG, not by IP, so a second copy of this notebook does not
# get its own budget. 9/min leaves a little headroom.
#
# That makes the FIRST run slow: 1192 photos at 9/min is roughly 2.2 hours. Every
# run after that only fetches photos that are new, so it takes a couple of minutes.
# Leave it running, or set MAX_PHOTOS to spread the first pull over several days.
MEDIA_PER_MINUTE = 9

GIT_PUSH = False     # set True to let the last cell commit and push the photos

## 2. Imports and helpers

In [ ]:
import configparser
import hashlib
import json
import os
import re
import subprocess
import sys
import time
import urllib.parse
from datetime import datetime, timezone

import geopandas as gpd
import pandas as pd
import requests

for d in (OUT_DIR, TABLE_DIR, PHOTO_DIR):
    d.mkdir(parents=True, exist_ok=True)

# A deleted or never-uploaded media file comes back as HTTP 200 plus this exact
# placeholder JPEG, identical across projects. Checked on every download.
EC5_PLACEHOLDER_MD5 = "4a383cffb6fb8bee91addb4a930d1e39"
EC5_PLACEHOLDER_SIZE = 40073

# Error codes the API returns in {"errors": [{"code": ..., "title": ...}]}.
EC5_ERROR_HELP = {
    "ec5_253": "Invalid client credentials - Client_ID or Client_Secret is wrong.",
    "ec5_254": "Token request rejected - a parameter is missing, usually Client_ID.",
    "ec5_256": "Access denied - private project and the token is missing, expired, "
               "or belongs to a different project.",
}

LOCATION_KEYS = ("latitude", "longitude", "accuracy",
                 "UTM_Northing", "UTM_Easting", "UTM_Zone")

# Epicollect5 declares a `datetime_format` per date/time question, in Java
# notation. Only the formats the form builder can produce are listed.
EC5_DATETIME_FORMATS = {
    "dd/MM/yyyy": "%d/%m/%Y",
    "MM/dd/yyyy": "%m/%d/%Y",
    "yyyy/MM/dd": "%Y/%m/%d",
    "dd/MM": "%d/%m",
    "MM/dd": "%m/%d",
    "MM/yyyy": "%m/%Y",
    "yyyy": "%Y",
}


class RateLimiter:
    """Spaces requests out so the published per-minute limits are never hit.

    `penalise` pushes the next allowed request further into the future, which is
    how a 429 or a low `X-RateLimit-Remaining` header is absorbed.
    """

    def __init__(self, per_minute, name=""):
        self.interval = 60.0 / float(per_minute)
        self.name = name
        self._next_at = 0.0

    def wait(self):
        delay = self._next_at - time.monotonic()
        if delay > 0:
            time.sleep(delay)
        self._next_at = time.monotonic() + self.interval

    def penalise(self, seconds):
        self._next_at = max(self._next_at, time.monotonic() + seconds)


def load_config(path):
    """Read config.ini into a lower-cased dict.

    The file has no `[section]` header, which configparser insists on, so one is
    added in memory when it is missing.
    """
    raw = Path(path).read_text(encoding="utf-8-sig")
    if not raw.lstrip().startswith("["):
        raw = "[epicollect5]\n" + raw
    parser = configparser.ConfigParser(interpolation=None)
    parser.read_string(raw)
    return {k.strip().lower(): v.strip()
            for section in parser.sections()
            for k, v in parser.items(section)}


def response_errors(resp):
    """Return the Epicollect5 error list carried by a response, if any."""
    if "json" not in resp.headers.get("Content-Type", ""):
        return []
    try:
        payload = resp.json()
    except ValueError:
        return []
    return payload.get("errors") or [] if isinstance(payload, dict) else []


def describe_errors(errors):
    return "; ".join(
        f"{e.get('code')} {e.get('title')} ({EC5_ERROR_HELP.get(e.get('code'), '')})".strip()
        for e in errors
    )


def sanitize_field_name(name, max_len=100):
    """Make a column name safe for ArcGIS.

    ArcGIS wants `[A-Za-z0-9_]` only, not starting with a digit. Long names keep
    their head *and* tail so questions that differ only in a trailing suffix stay
    distinct after truncation. Same rules as `table_to_geopackage.py`.
    """
    s = re.sub(r"[^A-Za-z0-9_]+", "_", str(name))
    s = re.sub(r"_+", "_", s).strip("_")
    if not s:
        s = "field"
    if s[0].isdigit():
        s = "f_" + s
    if max_len and len(s) > max_len:
        head_len = max_len // 2 - 1
        tail_len = max_len - head_len - 2
        s = s[:head_len].rstrip("_") + "__" + s[-tail_len:].lstrip("_")
    return s


def dedupe_columns_case_insensitive(cols, sep="_"):
    """Append `_N` to disambiguate names that collide once case is ignored."""
    cols = list(cols)
    original_lower = {c.lower() for c in cols}
    used, out = set(), []
    for c in cols:
        if c == "geometry":
            out.append(c)
            used.add(c.lower())
            continue
        candidate = c
        if candidate.lower() in used:
            n = 1
            while True:
                candidate = f"{c}{sep}{n}"
                if candidate.lower() not in used and candidate.lower() not in original_lower:
                    break
                n += 1
        out.append(candidate)
        used.add(candidate.lower())
    return out


print(f"python      {sys.version.split()[0]}")
print(f"pandas      {pd.__version__}")
print(f"geopandas   {gpd.__version__}")
print(f"output dir  {OUT_DIR}")
print(f"photo dir   {PHOTO_DIR}")

## 3. Read `config.ini`

Three values are needed, and all three are present:

```ini
Client_ID     = xxx
Client_Secret = xxxx
Project       = https://five.epicollect.net/api/export/project/xxxx
```

The file has no `[section]` header, which `configparser` normally requires, so
`load_config` inserts one in memory. Keys are matched case-insensitively.

Both credentials can instead come from the environment variables `EC5_CLIENT_ID`
and `EC5_CLIENT_SECRET`, which take priority. Use those if you ever want to share
this notebook without sharing the secret.

In [ ]:
cfg = load_config(CONFIG_PATH)

CLIENT_ID = (os.environ.get("EC5_CLIENT_ID") or cfg.get("client_id") or "").strip()
CLIENT_SECRET = (os.environ.get("EC5_CLIENT_SECRET") or cfg.get("client_secret") or "").strip()
PROJECT_URL = cfg.get("project", "").strip()

if not PROJECT_URL:
    # RuntimeError, not SystemExit: Jupyter swallows SystemExit and "Run All" would
    # carry on into cells that depend on values this one failed to produce.
    raise RuntimeError(f"No 'Project' entry found in {CONFIG_PATH}")

_parsed = urllib.parse.urlparse(PROJECT_URL)
API_BASE = f"{_parsed.scheme}://{_parsed.netloc}"
PROJECT_SLUG = _parsed.path.rstrip("/").split("/")[-1]

print(f"server        {API_BASE}")
print(f"project slug  {PROJECT_SLUG}")
print(f"client id     {CLIENT_ID or '** MISSING **'}")
print(f"client secret {'set (' + str(len(CLIENT_SECRET)) + ' chars)' if CLIENT_SECRET else '** MISSING **'}")

if not CLIENT_ID:
    print(
        "\nWARNING: Client_ID is not set, so only a PUBLIC project can be read.\n"
        "         This project answered 'ec5_256 Access denied' without a token,\n"
        "         which means it is private and a Client_ID is required."
    )

## 4. Authenticate

The token is written to `output/.ec5_token.json` and reused until it is within
five minutes of expiry. That matters: only **10 tokens per hour per IP** are
issued, which a few notebook re-runs would otherwise burn through.

In [ ]:
TOKEN_CACHE = OUT_DIR / ".ec5_token.json"
ACCESS_TOKEN = None

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "larvae-epicollect5-downloader/1.0"})

API_LIMITER = RateLimiter(ENTRIES_PER_MINUTE, "api")
MEDIA_LIMITER = RateLimiter(MEDIA_PER_MINUTE, "media")


def request_token(force=False):
    """Return a bearer token, reusing the cached one when it is still valid."""
    if not (CLIENT_ID and CLIENT_SECRET):
        return None

    if not force and TOKEN_CACHE.exists():
        try:
            cached = json.loads(TOKEN_CACHE.read_text(encoding="utf-8"))
            if (cached.get("slug") == PROJECT_SLUG
                    and cached.get("expires_at", 0) - 300 > time.time()):
                left = int(cached["expires_at"] - time.time())
                print(f"reusing cached token ({left // 60} min left)")
                return cached["access_token"]
        except (OSError, ValueError, KeyError):
            pass

    resp = SESSION.post(
        f"{API_BASE}/api/oauth/token",
        json={"grant_type": "client_credentials",
              "client_id": CLIENT_ID,
              "client_secret": CLIENT_SECRET},
        timeout=60,
    )
    errors = response_errors(resp)
    if errors:
        raise RuntimeError(f"Token request failed: {describe_errors(errors)}")
    resp.raise_for_status()
    payload = resp.json()

    token = payload["access_token"]
    expires_at = time.time() + float(payload.get("expires_in", 7200))
    try:
        TOKEN_CACHE.write_text(
            json.dumps({"slug": PROJECT_SLUG, "access_token": token,
                        "expires_at": expires_at}),
            encoding="utf-8",
        )
        if os.name != "nt":
            os.chmod(TOKEN_CACHE, 0o600)
    except OSError as exc:
        print(f"  (could not cache token: {exc})")
    print(f"new token issued, valid for {int(payload.get('expires_in', 7200)) // 60} min")
    return token


ACCESS_TOKEN = request_token()
print("authenticated" if ACCESS_TOKEN else "no token - continuing unauthenticated")

## 5. A polite, self-healing GET

Handles the three things that break a long download: the per-minute rate limit,
the two-hour token expiry, and transient network errors.

In [ ]:
def api_get(url, params=None, limiter=None, max_retries=6):
    """GET with throttling, 429 back-off and one automatic token refresh."""
    global ACCESS_TOKEN
    limiter = limiter or API_LIMITER
    refreshed = False
    last_problem = "unknown error"

    for attempt in range(max_retries):
        limiter.wait()
        headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"} if ACCESS_TOKEN else {}
        try:
            resp = SESSION.get(url, params=params, headers=headers, timeout=120)
        except requests.RequestException as exc:
            last_problem = f"{exc.__class__.__name__}: {exc}"
            wait = min(60, 2 ** attempt)
            print(f"    network error, retrying in {wait}s ({last_problem})")
            time.sleep(wait)
            continue

        errors = response_errors(resp)

        # Throttled. There is no Retry-After header on a 429 (checked live), and the
        # X-RateLimit-* headers are stripped from it too, so the wait is a fixed
        # escalating guess. The bucket is per minute, hence the 60s ceiling.
        if resp.status_code == 429 or any(e.get("code") == "ec5_255" for e in errors):
            wait = float(resp.headers.get("Retry-After") or 0) or min(60, 20 * (attempt + 1))
            print(f"    rate limited, sleeping {wait:.0f}s")
            limiter.penalise(wait)
            last_problem = "rate limited"
            continue

        # Slow down before the server has to say no.
        remaining = resp.headers.get("X-RateLimit-Remaining")
        if remaining is not None and remaining.isdigit() and int(remaining) <= 1:
            limiter.penalise(20)
        auth_problem = any(e.get("code") in ("ec5_256", "ec5_257") for e in errors)
        if (resp.status_code in (401, 403) or auth_problem) and not refreshed and CLIENT_ID:
            print("    token rejected, requesting a fresh one")
            ACCESS_TOKEN = request_token(force=True)
            refreshed = True
            last_problem = describe_errors(errors) or f"HTTP {resp.status_code}"
            continue

        if errors:
            raise RuntimeError(f"{url} -> {describe_errors(errors)}")
        if resp.status_code >= 500:
            last_problem = f"HTTP {resp.status_code}"
            wait = min(60, 5 * (attempt + 1))
            print(f"    server error {resp.status_code}, retrying in {wait}s")
            time.sleep(wait)
            continue

        resp.raise_for_status()
        return resp

    raise RuntimeError(f"Gave up on {url} after {max_retries} attempts: {last_problem}")

## 6. Project structure

The project definition tells us which forms exist, which questions are photos or
locations, which questions are branches, and what each question is called in the
exported table.

In [ ]:
definition = api_get(f"{API_BASE}/api/export/project/{PROJECT_SLUG}").json()
project = definition["data"]["project"]
meta = definition.get("meta", {})

(OUT_DIR / "project_definition.json").write_text(
    json.dumps(definition, indent=1, ensure_ascii=False), encoding="utf-8"
)

print(f"project : {project['name']}  ({project['slug']})")
stats = meta.get("project_stats", {})
print(f"entries : {stats.get('total_entries', '?')}")
print(f"files   : {stats.get('total_files', '?')}  "
      f"({round(stats.get('total_bytes', 0) / 1e6, 1)} MB)")
print(f"forms   : {len(project.get('forms', []))}")

In [ ]:
def flatten_mapping(mapping_forms):
    """Flatten one project mapping into {input_ref: exported column name}."""
    out = {}

    def walk(node):
        if not isinstance(node, dict):
            return
        for ref, spec in node.items():
            if not isinstance(spec, dict):
                continue
            if "map_to" in spec:
                out[ref] = spec["map_to"]
            walk(spec.get("branch"))
            walk(spec.get("group"))

    for inputs in (mapping_forms or {}).values():
        walk(inputs)
    return out


def iter_inputs(inputs, owner_ref, owner_kind):
    """Walk a form's questions, descending into groups and branches.

    `owner_kind` is "form" for questions answered on the form row itself and
    "branch" for questions inside a branch, whose answers are fetched separately.
    """
    for inp in inputs or []:
        if not isinstance(inp, dict):
            continue
        yield inp, owner_ref, owner_kind
        for child in inp.get("group") or []:
            yield child, owner_ref, owner_kind
        if inp.get("type") == "branch":
            for child in inp.get("branch") or []:
                yield child, inp["ref"], "branch"
                for grandchild in child.get("group") or []:
                    yield grandchild, inp["ref"], "branch"


mappings = meta.get("project_mapping", []) or []
chosen = next((m for m in mappings if m.get("map_index") == MAP_INDEX), None)
if chosen is None:
    chosen = next((m for m in mappings if m.get("is_default")), mappings[0] if mappings else {})
    print(f"map_index {MAP_INDEX} not found, using '{chosen.get('name')}' "
          f"(map_index {chosen.get('map_index')}) instead")
MAP_INDEX = chosen.get("map_index", 0)
map_to = flatten_mapping(chosen.get("forms"))
print(f"mapping : {chosen.get('name')} (map_index {MAP_INDEX})")

rows = []
for form_index, form in enumerate(project.get("forms", [])):
    for inp, owner_ref, owner_kind in iter_inputs(form.get("inputs"), form["ref"], "form"):
        rows.append({
            "form_index": form_index,
            "form_ref": form["ref"],
            "form_name": form.get("name") or form.get("slug") or f"form{form_index}",
            "owner_kind": owner_kind,
            "owner_ref": owner_ref,
            "input_ref": inp["ref"],
            "type": inp.get("type"),
            "question": inp.get("question"),
            "datetime_format": inp.get("datetime_format"),
            "column": map_to.get(inp["ref"], inp["ref"]),
        })

fields = pd.DataFrame(rows)
fields.to_csv(OUT_DIR / "field_catalogue.csv", index=False, encoding="utf-8-sig")

print(f"\n{len(fields)} questions across {fields['form_ref'].nunique()} form(s)")
print(fields["type"].value_counts().to_string())
fields.head(20)

## 7. Download every entry

One request set per form, plus one per branch. Sorted by `created_at` ascending so
that entries uploaded while the download is running are appended at the end rather
than shifting rows between pages.

In [ ]:
def fetch_entries(form_ref, branch_ref=None):
    """Page through the entries endpoint and return a list of entry dicts."""
    url = f"{API_BASE}/api/export/entries/{PROJECT_SLUG}"
    params = {
        "form_ref": form_ref,
        "per_page": PER_PAGE,
        "map_index": MAP_INDEX,
        "format": "json",
        "sort_by": "created_at",
        "sort_order": "ASC",
    }
    if branch_ref:
        params["branch_ref"] = branch_ref

    entries, page, last_page = [], 1, 1
    while page <= last_page:
        payload = api_get(url, params={**params, "page": page}).json()
        page_meta = payload.get("meta", {})
        last_page = int(page_meta.get("last_page") or 1)
        batch = payload.get("data", {}).get("entries", []) or []
        entries.extend(batch)
        total = page_meta.get("total", "?")
        print(f"    page {page}/{last_page}  +{len(batch):>4}  ({len(entries)}/{total})")
        if not batch:
            break
        page += 1
    return entries


# Every table to pull: each form, then each branch question inside each form.
targets = []
for form_index, form in enumerate(project.get("forms", [])):
    targets.append({
        "key": form.get("slug") or f"form{form_index}",
        "label": form.get("name") or f"form{form_index}",
        "kind": "form",
        "form_ref": form["ref"],
        "branch_ref": None,
        "form_index": form_index,
    })

branch_fields = fields[fields["type"] == "branch"] if not fields.empty else pd.DataFrame()
for _, br in branch_fields.iterrows():
    targets.append({
        "key": f"{br['form_name']}__{br['question'] or br['input_ref']}",
        "label": f"{br['form_name']} / {br['question']} (branch)",
        "kind": "branch",
        "form_ref": br["form_ref"],
        "branch_ref": br["input_ref"],
        "form_index": br["form_index"],
    })

# One collision-free, ArcGIS-safe name per table, reused for the CSV filename, the
# Excel sheet and the GeoPackage layer so the three always line up. Two branches with
# similar question text would otherwise sanitise to the same name and silently
# overwrite one another.
_used_names = set()
for target in targets:
    name = sanitize_field_name(target["key"], max_len=60)
    if name.lower() in _used_names:
        n = 2
        while f"{name}_{n}".lower() in _used_names:
            n += 1
        name = f"{name}_{n}"
    _used_names.add(name.lower())
    target["safe_name"] = name

raw_tables = {}
for target in targets:
    print(f"\n{target['label']}")
    entries = fetch_entries(target["form_ref"], target["branch_ref"])
    raw_tables[target["key"]] = pd.DataFrame(entries)
    target["n_rows"] = len(entries)

print("\n" + "-" * 60)
for target in targets:
    print(f"{target['n_rows']:>7} rows   {target['label']}")

## 8. Tidy the tables

* location objects are expanded into `<question>_latitude`, `_longitude`, `_accuracy`, `_UTM_*`
* `created_at` / `uploaded_at` become real UTC timestamps
* date questions are parsed using the format the project declares for them
* checkbox / multi-answer values become a readable `"a; b"` string

In [ ]:
def expand_locations(df, columns):
    """Replace each location column (a nested object) with flat numeric columns."""
    for col in columns:
        if col not in df.columns:
            continue
        values = df[col]
        for key in LOCATION_KEYS:
            df[f"{col}_{key}"] = values.map(
                lambda v, k=key: v.get(k) if isinstance(v, dict) else None
            )
        for key in ("latitude", "longitude", "accuracy", "UTM_Northing", "UTM_Easting"):
            df[f"{col}_{key}"] = pd.to_numeric(df[f"{col}_{key}"], errors="coerce")
        df = df.drop(columns=[col])
    return df


def parse_declared_dates(df, date_fields):
    """Parse date questions with the format declared in the project definition."""
    for col, fmt in date_fields.items():
        if col not in df.columns:
            continue
        pattern = EC5_DATETIME_FORMATS.get(fmt)
        source = df[col].astype("string")
        filled = source.notna() & (source.str.strip() != "")
        if not filled.any():
            continue
        parsed = pd.to_datetime(source, format=pattern, errors="coerce") if pattern \
            else pd.to_datetime(source, dayfirst=True, errors="coerce")
        ok = parsed.notna() & filled
        if ok.sum() >= 0.9 * filled.sum():
            df[col] = parsed
        else:
            print(f"    kept '{col}' as text ({ok.sum()}/{filled.sum()} parsed as {fmt})")
    return df


def coerce_declared_numerics(df, numeric_fields):
    """Force integer and decimal questions to a numeric dtype.

    An unanswered numeric question comes back as "" rather than null, so a single
    blank answer turns the whole column into text. Left alone, a question that is
    fully answered today and partly blank tomorrow would change type between runs
    and break the ArcGIS Online overwrite. The nullable Int64/Float64 dtypes keep
    the GeoPackage column type identical whether or not blanks are present.
    """
    for col, kind in numeric_fields.items():
        if col not in df.columns:
            continue
        values = pd.to_numeric(df[col].replace("", pd.NA), errors="coerce")
        df[col] = values.astype("Float64" if kind == "decimal" else "Int64")
    return df


def stringify_containers(df):
    """Flatten any list/dict answers into text so they survive CSV and GPKG."""
    for col in df.columns:
        if df[col].dtype != object:
            continue
        if df[col].map(lambda v: isinstance(v, (list, dict))).any():
            df[col] = df[col].map(
                lambda v: "; ".join(str(x) for x in v) if isinstance(v, list)
                else json.dumps(v, ensure_ascii=False) if isinstance(v, dict)
                else v
            )
    return df


tables = {}
for target in targets:
    key = target["key"]
    df = raw_tables[key].copy()
    if df.empty:
        tables[key] = df
        continue

    scope = fields[fields["form_ref"] == target["form_ref"]]
    scope = scope[scope["owner_kind"] == ("branch" if target["kind"] == "branch" else "form")]
    if target["kind"] == "branch":
        scope = scope[scope["owner_ref"] == target["branch_ref"]]

    df = expand_locations(df, scope.loc[scope["type"] == "location", "column"].tolist())
    df = parse_declared_dates(
        df,
        dict(zip(scope.loc[scope["type"] == "date", "column"],
                 scope.loc[scope["type"] == "date", "datetime_format"])),
    )
    numeric = scope[scope["type"].isin(("integer", "decimal"))]
    df = coerce_declared_numerics(df, dict(zip(numeric["column"], numeric["type"])))
    for col in ("created_at", "uploaded_at"):
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce", utc=True, format="ISO8601")
    df = stringify_containers(df)

    target["photo_columns"] = scope.loc[scope["type"] == "photo", "column"].tolist()
    target["location_columns"] = scope.loc[scope["type"] == "location", "column"].tolist()
    tables[key] = df
    print(f"{key:<40} {len(df):>6} rows x {len(df.columns):>3} cols  "
          f"photos={len(target['photo_columns'])} locations={len(target['location_columns'])}")

## 9. Download the photos

Filenames are globally unique (`<entry uuid>_<timestamp>.jpg`) so a single flat
folder is fine. Files already present are skipped, which is what makes the daily
run fast, and each file is written to `.part` first so an interrupted run never
leaves a truncated image behind.

In [ ]:
def media_filename(value):
    """Pull the `name` parameter out of a media URL (or accept a bare filename).

    The result is used as a path, so anything that could escape PHOTO_DIR or is
    illegal on Windows is rejected rather than sanitised - a mangled name would
    not match the file the server has anyway.
    """
    if not isinstance(value, str) or not value.strip():
        return ""
    value = value.strip()
    query = urllib.parse.parse_qs(urllib.parse.urlparse(value).query)
    name = (query.get("name") or [""])[0] or value.split("/")[-1]
    name = urllib.parse.unquote(name).strip()
    if not name or name in (".", ".."):
        return ""
    if set(name) & set('\\/:*?"<>|') or "\x00" in name:
        print(f"    skipping unusable media name: {name!r}")
        return ""
    return name


def download_photo(filename):
    """Download one photo. Returns one of skip / ok / missing / error."""
    dest = PHOTO_DIR / filename
    if dest.exists() and dest.stat().st_size > 0:
        return "skip", dest
    try:
        resp = api_get(
            f"{API_BASE}/api/export/media/{PROJECT_SLUG}",
            params={"type": "photo", "format": PHOTO_FORMAT, "name": filename},
            limiter=MEDIA_LIMITER,
        )
    except (RuntimeError, requests.RequestException) as exc:
        print(f"    error {filename}: {exc}")
        return "error", None

    # A photo that was never synced off the phone still answers 200 with a
    # "PHOTO NOT SYNCED YET" placeholder. Two independent signals: the server marks
    # the placeholder no-store while real files are immutable, and the placeholder
    # bytes are always identical.
    body = resp.content
    cache_control = resp.headers.get("Cache-Control", "")
    if ("no-store" in cache_control
            or (len(body) == EC5_PLACEHOLDER_SIZE
                and hashlib.md5(body).hexdigest() == EC5_PLACEHOLDER_MD5)):
        return "missing", None
    if not resp.headers.get("Content-Type", "").startswith("image/"):
        print(f"    not an image: {filename} ({resp.headers.get('Content-Type')})")
        return "error", None

    part = dest.with_name(dest.name + ".part")
    part.write_bytes(body)
    part.replace(dest)
    return "ok", dest


# Every photo referenced anywhere in the project. Filenames are globally unique
# (<entry uuid>_<timestamp>.jpg), so the same file is never fetched twice even when
# two questions happen to point at it.
wanted = set()
for target in targets:
    df = tables[target["key"]]
    for col in target.get("photo_columns", []):
        if col in df.columns:
            wanted.update(n for n in df[col].map(media_filename) if n)
wanted = sorted(wanted)
already = {p.name for p in PHOTO_DIR.glob("*") if p.is_file() and not p.name.endswith(".part")}
outstanding = [n for n in wanted if n not in already]
todo = outstanding[:MAX_PHOTOS] if MAX_PHOTOS is not None else outstanding

def ensure_fresh_token(min_seconds=600):
    """Refresh before expiry rather than after a request has already failed.

    A large first download runs for hours while the token only lasts two, and a
    reactive refresh costs a wasted request out of a 10-per-minute budget.
    """
    global ACCESS_TOKEN
    if not (TOKEN_CACHE.exists() and CLIENT_ID):
        return
    try:
        cached = json.loads(TOKEN_CACHE.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return
    if cached.get("expires_at", 0) - min_seconds <= time.time():
        ACCESS_TOKEN = request_token(force=True)


print(f"{len(wanted)} photos referenced, "
      f"{len(wanted) - len(outstanding)} already on disk, "
      f"{len(outstanding)} still missing")
if len(todo) != len(outstanding):
    print(f"  fetching only {len(todo)} of them (MAX_PHOTOS = {MAX_PHOTOS})")
if DOWNLOAD_PHOTOS and todo:
    eta = len(todo) / MEDIA_PER_MINUTE
    shown = f"{eta / 60:.1f} hours" if eta > 90 else f"{eta:.0f} minutes"
    print(f"  at {MEDIA_PER_MINUTE}/min this will take about {shown}. "
          f"Interrupting is safe, the next run picks up where this one stopped.")

results = {}
if DOWNLOAD_PHOTOS and todo:
    started = time.time()
    for i, name in enumerate(todo, 1):
        if i % 50 == 0:
            ensure_fresh_token()
        status, _ = download_photo(name)
        results[name] = status
        if i % 25 == 0 or i == len(todo):
            elapsed = time.time() - started
            left = (len(todo) - i) * elapsed / max(i, 1) / 60
            print(f"  [{i}/{len(todo)}] {i / max(elapsed, 1e-6) * 60:.1f}/min  "
                  f"ok={sum(v == 'ok' for v in results.values())} "
                  f"missing={sum(v == 'missing' for v in results.values())} "
                  f"error={sum(v == 'error' for v in results.values())}  "
                  f"~{left:.0f} min left")
elif not DOWNLOAD_PHOTOS:
    print("DOWNLOAD_PHOTOS is False, skipping")

never_synced = sorted(n for n, v in results.items() if v == "missing")
if never_synced:
    (OUT_DIR / "photos_not_synced.txt").write_text("\n".join(never_synced), encoding="utf-8")
    print(f"\n{len(never_synced)} photo(s) are referenced by an entry but were never "
          f"synced off the device; listed in photos_not_synced.txt")

on_disk = {p.name for p in PHOTO_DIR.glob("*") if p.is_file() and not p.name.endswith(".part")}
print(f"\n{len(on_disk)} photos in {PHOTO_DIR}")

## 10. Publish the photos to GitHub

The URLs written in the next cell name a GitHub repository, so the photos have to
be *in* that repository before the URLs can be true. Publishing therefore happens
first, and the URL cell then links only to files it can see on the branch.

Set `GITHUB_REPO_DIR` to a local clone with a remote and stored credentials, and
`GIT_PUSH = True`. The clone is checked against `GITHUB_OWNER` / `GITHUB_REPO` /
`GITHUB_BRANCH`, because a clone of the wrong repository would produce URLs that
look perfectly valid and 404 for every viewer.

In [ ]:
def git(*args, cwd):
    done = subprocess.run(["git", *args], cwd=str(cwd), capture_output=True, text=True)
    if done.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{done.stderr.strip()}")
    return done.stdout.strip()


published = set()

if not GITHUB_REPO_DIR:
    print("GITHUB_REPO_DIR is not set.\n"
          f"  Photos are being kept locally in {PHOTO_DIR}, which is not the\n"
          f"  repository the URLs name, so NO photo URLs will be written and the\n"
          "  ArcGIS popup will have no images.\n"
          "  To fix: clone the photo repository, point GITHUB_REPO_DIR at it, set\n"
          "  GIT_PUSH = True, and re-run from the settings cell.")
else:
    repo = Path(GITHUB_REPO_DIR)
    remote = git("remote", "get-url", "origin", cwd=repo)
    expected = f"{GITHUB_OWNER}/{GITHUB_REPO}"
    if expected.lower() not in remote.lower().replace(".git", ""):
        raise RuntimeError(
            f"{repo}\n  has origin {remote}\n  but the photo URLs name {expected}. "
            "Fix GITHUB_OWNER / GITHUB_REPO / GITHUB_REPO_DIR before uploading."
        )
    on_branch = git("rev-parse", "--abbrev-ref", "HEAD", cwd=repo)
    if on_branch != GITHUB_BRANCH:
        raise RuntimeError(
            f"{repo} is on branch '{on_branch}' but the photo URLs name "
            f"'{GITHUB_BRANCH}'. Check out {GITHUB_BRANCH} or change GITHUB_BRANCH."
        )

    if GIT_PUSH:
        # Stage first, then count: `git status --porcelain` collapses a whole
        # untracked directory into a single line, so counting before the add would
        # report "1 file" for a thousand new photos.
        git("add", "--", GITHUB_SUBDIR, cwd=repo)
        staged = git("diff", "--cached", "--name-only", "--", GITHUB_SUBDIR, cwd=repo)
        n_staged = len([line for line in staged.splitlines() if line.strip()])
        if n_staged:
            stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d")
            git("commit", "-m",
                f"Add Epicollect5 photos ({stamp}, {n_staged} file(s))", cwd=repo)
            print(f"committed {n_staged} file(s)")
        else:
            print("no new photos to commit")
        git("push", "origin", GITHUB_BRANCH, cwd=repo)
        print(f"pushed to {expected} ({GITHUB_BRANCH})")
    else:
        git("fetch", "origin", GITHUB_BRANCH, cwd=repo)
        print("GIT_PUSH is False; only photos already on the remote will get a URL")

    # What is genuinely reachable on the branch the URLs point at.
    listing = git("ls-tree", "-r", "--name-only", f"origin/{GITHUB_BRANCH}",
                  "--", GITHUB_SUBDIR, cwd=repo)
    published = {line.rsplit("/", 1)[-1] for line in listing.splitlines() if line.strip()}

print(f"\n{len(published)} photo(s) reachable on GitHub")

## 11. Public photo URLs for the ArcGIS popup

One `<question>_url` column per photo question. A URL is written only for a photo
that is actually on the GitHub branch, so the popup never shows a broken image; the
`<question>_file` column always holds the filename regardless, for tracing.

In the ArcGIS Online popup configuration add an **Image** media element and set its
URL to `{<question>_url}`.

In [ ]:
def github_photo_url(filename):
    """Public URL for a photo, or None if it is not published yet."""
    if not filename or filename not in published:
        return None
    subdir = GITHUB_SUBDIR.strip("/")
    prefix = f"{subdir}/" if subdir else ""
    safe = urllib.parse.quote(filename)
    if PHOTO_URL_STYLE == "pages":
        return f"https://{GITHUB_OWNER}.github.io/{GITHUB_REPO}/{prefix}{safe}"
    return (f"https://raw.githubusercontent.com/{GITHUB_OWNER}/{GITHUB_REPO}/"
            f"{GITHUB_BRANCH}/{prefix}{safe}")


referenced = linked = 0
for target in targets:
    df = tables[target["key"]]
    for col in target.get("photo_columns", []):
        if col not in df.columns:
            continue
        names = df[col].map(media_filename)
        df[f"{col}_file"] = names.replace("", None)
        df[f"{col}_url"] = names.map(github_photo_url)
        referenced += int(df[f"{col}_file"].notna().sum())
        linked += int(df[f"{col}_url"].notna().sum())

print(f"{linked} of {referenced} photo references have a public URL")
if linked < referenced:
    print(f"  {referenced - linked} have no URL because the file is not on the "
          f"{GITHUB_BRANCH} branch yet. Their popups will show no image.")
if published:
    print("example:", github_photo_url(sorted(published)[0]))

## 12. Save the CSVs

In [ ]:
for target in targets:
    df = tables[target["key"]]
    path = TABLE_DIR / f"{target['safe_name']}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"{len(df):>7} rows -> {path.name}")

with pd.ExcelWriter(OUT_DIR / "epicollect5_tables.xlsx", engine="openpyxl") as writer:
    used_sheets = set()
    for target in targets:
        df = tables[target["key"]].copy()
        for col in df.columns:
            if isinstance(df[col].dtype, pd.DatetimeTZDtype):
                df[col] = df[col].dt.tz_localize(None)   # Excel cannot store tz-aware
        # Excel caps sheet names at 31 characters, so two long form names can collide
        # and silently overwrite each other.
        sheet = target["safe_name"][:31]
        if sheet.lower() in used_sheets:
            n = 2
            while f"{sheet[:28]}_{n}".lower() in used_sheets:
                n += 1
            sheet = f"{sheet[:28]}_{n}"
        used_sheets.add(sheet.lower())
        df.to_excel(writer, sheet_name=sheet, index=False)
print(f"\nworkbook -> {OUT_DIR / 'epicollect5_tables.xlsx'}")

## 13. Build the GeoPackage

One layer per table. Tables that carry a location question become point layers in
EPSG:4326; the rest are written as plain (aspatial) tables so the joins are still
available in ArcGIS. Field names are made ArcGIS-safe and the file is rebuilt from
scratch on every run.

In [ ]:
import pyogrio


def arcgis_ready(df):
    """Cast to types that survive GeoPackage -> ArcGIS Online.

    Columns are deliberately *not* dropped unless `DROP_EMPTY_COLUMNS` is on. The
    daily upload relies on overwriting the hosted layer, and that only works while
    the field list stays identical from one day to the next.
    """
    out = df.copy()
    for col in out.columns:
        if col == "geometry" or pd.api.types.is_datetime64_any_dtype(out[col]):
            continue
        if out[col].dtype == object:
            out[col] = out[col].astype("string")
    drop = []
    if DROP_EMPTY_COLUMNS:
        drop = [c for c in out.columns
                if c != "geometry" and (out[c].isna().all()
                                        or (out[c].dtype == "string"
                                            and not out[c].dropna().str.strip().ne("").any()))]
        out = out.drop(columns=drop)
    out.columns = [c if c == "geometry" else sanitize_field_name(c) for c in out.columns]
    out.columns = dedupe_columns_case_insensitive(out.columns)
    return out, drop


if GPKG_PATH.exists():
    try:
        GPKG_PATH.unlink()
    except OSError as exc:
        raise RuntimeError(
            f"Cannot replace {GPKG_PATH} ({exc}).\n"
            "Close it in ArcGIS Pro / QGIS first, then re-run this cell."
        )

layer_report = []
for target in targets:
    df = tables[target["key"]]
    layer = target["safe_name"]
    if df.empty:
        layer_report.append((layer, 0, "empty, skipped"))
        continue

    # Geometry comes from the first location question that actually has coordinates.
    lat_col = lon_col = None
    for loc in target.get("location_columns", []):
        lat, lon = f"{loc}_latitude", f"{loc}_longitude"
        if lat in df.columns and df[lat].notna().any() and df[lon].notna().any():
            lat_col, lon_col = lat, lon
            break

    if lat_col is None:
        clean, _ = arcgis_ready(df)
        pyogrio.write_dataframe(clean, GPKG_PATH, layer=layer, driver="GPKG",
                                append=GPKG_PATH.exists())
        layer_report.append((layer, len(clean), "table (no location question)"))
        continue

    lat = df[lat_col]
    lon = df[lon_col]
    valid = lat.between(-90, 90) & lon.between(-180, 180) & lat.notna() & lon.notna()
    valid &= ~((lat == 0) & (lon == 0))          # null island, an unfixed GPS reading
    sub = df.loc[valid].copy()
    if sub.empty:
        clean, _ = arcgis_ready(df)
        pyogrio.write_dataframe(clean, GPKG_PATH, layer=layer, driver="GPKG",
                                append=GPKG_PATH.exists())
        layer_report.append((layer, len(clean), "table (no valid coordinates)"))
        continue

    clean, dropped = arcgis_ready(sub)
    gdf = gpd.GeoDataFrame(
        clean,
        geometry=gpd.points_from_xy(lon.loc[valid], lat.loc[valid]),
        crs="EPSG:4326",
    )
    gdf.to_file(GPKG_PATH, layer=layer, driver="GPKG",
                mode="a" if GPKG_PATH.exists() else "w")
    skipped = len(df) - len(sub)
    note = f"points from {lat_col[:-9]}"
    if skipped:
        note += f", {skipped} row(s) without usable coordinates dropped"
    layer_report.append((layer, len(gdf), note))

print(f"{GPKG_PATH}\n")
for layer, n, note in layer_report:
    print(f"  {layer:<45} {n:>6}  {note}")
print("\nlayers in file:", [row[0] for row in pyogrio.list_layers(GPKG_PATH)])

## 14. Run summary

In [ ]:
print(f"run finished {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"project      {project['name']} ({PROJECT_SLUG})")
print()
for target in targets:
    print(f"  {target['n_rows']:>7} rows   {target['label']}")
print()
print(f"  {len(on_disk):>7} photos on disk in {PHOTO_DIR}")
print(f"  {len(published):>7} photos published on GitHub")
if len(wanted) > len(on_disk):
    print(f"  {len(wanted) - len(on_disk):>7} still to download "
          f"(~{(len(wanted) - len(on_disk)) / MEDIA_PER_MINUTE / 60:.1f} h) - "
          f"just run this notebook again")
if results:
    for status in ("ok", "missing", "error"):
        n = sum(v == status for v in results.values())
        if n:
            print(f"  {n:>7} {status} this run")
print()
print(f"  tables     {TABLE_DIR}")
print(f"  geopackage {GPKG_PATH}")

if linked < referenced:
    print(f"\nNOTE: {referenced - linked} photo reference(s) have no public URL yet, so "
          f"those popups will have no image.")
if not GITHUB_REPO_DIR:
    print("NOTE: GITHUB_REPO_DIR is unset, so no photo URLs were written at all.")

## One-time setup

1. Create the GitHub repository that will hold the photos and clone it locally.
2. In the settings cell set `GITHUB_REPO_DIR` to that clone, set `GITHUB_OWNER` and
   `GITHUB_REPO` to match, and set `GIT_PUSH = True`.
3. Run **Cell -> Run All**. The first run downloads all 1192 photos at the API's
   10 per minute, so allow **about two hours**. Stopping it is safe: the next run
   continues where it left off. Setting `MAX_PHOTOS` spreads it over several days.
4. Upload `epicollect5.gpkg` to ArcGIS Online and publish it as a hosted feature
   layer. In the popup add an **Image** media element whose URL is
   `{<question>_url}`, for example `{f_6_Surrounding_url}`. The field names printed
   by the GeoPackage cell are the exact sanitised ones.

## Daily routine

1. Run **Cell -> Run All**. Only new photos are fetched, so this takes a couple of
   minutes, and the photos are committed and pushed for you.
2. Upload `epicollect5.gpkg` and **overwrite** the existing hosted feature layer.
   Overwriting keeps the item id, the popup configuration and every web map that
   points at it. Replacing the item instead would break all of them.

### Things that will bite

* **Overwriting only works while the schema is stable.** Adding a question in
  Epicollect5 adds a column and ArcGIS Online will refuse the overwrite until the
  layer is republished. `MAP_INDEX = 0`, `DROP_EMPTY_COLUMNS = False` and the
  declared-type coercion above all exist to keep the field list identical between
  runs.
* **A photo only gets a URL once it is on GitHub.** If a run downloads photos but
  the push fails, those rows get no URL and their popups show no image. The summary
  says so explicitly.
* `raw.githubusercontent.com` is fine for this volume (about 150 MB), but it is not
  a CDN. If popups start loading slowly, switch on GitHub Pages for the repository
  and set `PHOTO_URL_STYLE = "pages"`.
* The token cache and the downloaded tables live under `OUT_DIR`, which is
  deliberately outside the photo repository so credentials are never committed.